# 09. 시간대별 탑승건수 전처리

`탑승내역_정제`의 `승차일시`를 기준으로 2025년 실제 탑승을 일자·시간대별로 집계한다. 모든 날짜 × 24시간 조합을 만든 뒤 관측되지 않은 시간대는 0건으로 채운다.

In [ ]:
from pathlib import Path
import pandas as pd

## 0. 경로 및 집계 기간 설정

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / '서울시설공단_장애인콜택시 탑승내역_정제_20251231.csv'
REFERENCE_PATH = PROJECT_ROOT / 'data' / 'raw' / '서울시설공단_장애인콜택시 시간대별 탑승건수_20251231.csv'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / '서울시설공단_장애인콜택시 시간대별 탑승건수_정제_20251231.csv'
START_DATE = pd.Timestamp('2025-01-01')
END_DATE = pd.Timestamp('2025-12-31')

In [ ]:
print(f'프로젝트 루트: {PROJECT_ROOT}')
print(f'입력 파일: {INPUT_PATH}')
print(f'출력 파일: {OUTPUT_PATH}')

## 1. 정제된 탑승내역 불러오기

In [ ]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(f'입력 파일이 없습니다: {INPUT_PATH}')

df_raw = pd.read_csv(INPUT_PATH, usecols=['승차일시'])
print(f'원본 행 수: {len(df_raw):,}')
df_raw.head()

### 승차일시 자료형 변환 및 결측 확인

In [ ]:
df_raw['승차일시'] = pd.to_datetime(df_raw['승차일시'], errors='coerce')

load_summary = pd.Series({
    '전체 행 수': len(df_raw),
    '승차일시 정상 행 수': df_raw['승차일시'].notna().sum(),
    '승차일시 결측/변환 실패': df_raw['승차일시'].isna().sum(),
})
display(load_summary.to_frame('값'))
print(f"승차일시 범위: {df_raw['승차일시'].min()} ~ {df_raw['승차일시'].max()}")

## 2. 승차일시 결측치 처리

탑승건수는 실제 `승차일시`를 기준으로 집계하므로 `승차일시`가 결측인 행은 시간대를 결정할 수 없다. 임의의 시각으로 대체하면 시간대별 건수가 왜곡되므로 대체하지 않고 집계 대상에서 제외한다. 원본 데이터는 변경하지 않으며 결측 행을 별도 데이터프레임에 보관한다.

In [ ]:
df_missing_ride_time = df_raw.loc[df_raw['승차일시'].isna()].copy()

print(f'승차일시 결측 행: {len(df_missing_ride_time):,}건')
print(f'전체 대비 결측률: {len(df_missing_ride_time) / len(df_raw) * 100:.2f}%')
df_missing_ride_time.head()

### 승차일시 결측 행 제외 및 처리 결과 검증

In [ ]:
before_missing_handling = len(df_raw)
df_boarded = df_raw.dropna(subset=['승차일시']).copy()
after_missing_handling = len(df_boarded)

missing_handling_summary = pd.Series({
    '처리 전 행 수': before_missing_handling,
    '제외된 결측 행 수': before_missing_handling - after_missing_handling,
    '처리 후 행 수': after_missing_handling,
    '처리 후 승차일시 결측 수': df_boarded['승차일시'].isna().sum(),
})
display(missing_handling_summary.to_frame('값'))

assert before_missing_handling - after_missing_handling == len(df_missing_ride_time)
assert df_boarded['승차일시'].isna().sum() == 0
print('승차일시 결측치 처리 검증 통과')

## 3. 실제 탑승 선별

결측치 처리 후 승차일 기준 2025년 데이터만 집계 대상으로 사용한다.

### 일자·시간 파생 컬럼 생성

In [ ]:
df_boarded['일자'] = df_boarded['승차일시'].dt.normalize()
df_boarded['시간'] = df_boarded['승차일시'].dt.hour
df_boarded.head()

### 2025년 집계 대상 선별

In [ ]:
period_mask = df_boarded['일자'].between(START_DATE, END_DATE)
df_outside_period = df_boarded.loc[~period_mask].copy()
df_boarded = df_boarded.loc[period_mask].copy()

print(f'2025년 집계 대상: {len(df_boarded):,}건')
print(f'기간 밖 제외: {len(df_outside_period):,}건')
df_outside_period.head()

## 4. 일자·시간대별 집계 및 누락 시간 보완

In [ ]:
boarding_counts = (
    df_boarded.groupby(['일자', '시간'], as_index=False)
    .size()
    .rename(columns={'size': '탑승건수'})
)
boarding_counts.head()

### 365일 × 24시간 전체 기준표 생성

In [ ]:
full_index = pd.MultiIndex.from_product(
    [pd.date_range(START_DATE, END_DATE, freq='D'), range(24)],
    names=['일자', '시간'],
)
print(f'전체 기준 조합 수: {len(full_index):,}')

### 누락 시간대를 0건으로 보완하고 출력 형식 정리

In [ ]:
df_boarding_hour = (
    boarding_counts.set_index(['일자', '시간'])
    .reindex(full_index, fill_value=0)
    .reset_index()
)
df_boarding_hour.insert(0, '시설명', '서울시설공단')
df_boarding_hour['시간대'] = df_boarding_hour['시간'].map(lambda x: f'{x:02d}:00:00')
df_boarding_hour = df_boarding_hour[['시설명', '일자', '시간대', '탑승건수']]
df_boarding_hour['탑승건수'] = df_boarding_hour['탑승건수'].astype('int64')
df_boarding_hour.head(24)

## 5. 품질 검증

In [ ]:
expected_rows = 365 * 24
quality_summary = pd.Series({
    '예상 행 수': expected_rows,
    '실제 행 수': len(df_boarding_hour),
    '일자 수': df_boarding_hour['일자'].nunique(),
    '시간대 수': df_boarding_hour['시간대'].nunique(),
    '중복 키 수': df_boarding_hour.duplicated(['시설명', '일자', '시간대']).sum(),
    '결측값 수': df_boarding_hour.isna().sum().sum(),
    '음수 행 수': (df_boarding_hour['탑승건수'] < 0).sum(),
    '0건 시간대 수': (df_boarding_hour['탑승건수'] == 0).sum(),
    '총 탑승건수': df_boarding_hour['탑승건수'].sum(),
})
display(quality_summary.to_frame('값'))

assert len(df_boarding_hour) == expected_rows
assert df_boarding_hour.duplicated(['시설명', '일자', '시간대']).sum() == 0
assert df_boarding_hour.isna().sum().sum() == 0
assert (df_boarding_hour['탑승건수'] >= 0).all()
assert df_boarding_hour['탑승건수'].sum() == len(df_boarded)
print('품질 검증 통과')

## 6. 제공된 시간대별 탑승건수와 비교

제공 데이터와 정제 탑승내역의 집계 기준이 다를 수 있으므로 차이는 검증 정보로만 사용한다.

In [ ]:
df_reference = pd.read_csv(REFERENCE_PATH)
df_reference['일자'] = pd.to_datetime(df_reference['일자'], errors='coerce')

comparison = df_boarding_hour.merge(
    df_reference,
    on=['시설명', '일자', '시간대'],
    how='outer',
    suffixes=('_정제', '_제공'),
    indicator=True,
)
comparison['차이(정제-제공)'] = (
    comparison['탑승건수_정제'].fillna(0) - comparison['탑승건수_제공'].fillna(0)
)

comparison_summary = pd.Series({
    '정제 총 탑승건수': comparison['탑승건수_정제'].fillna(0).sum(),
    '제공 총 탑승건수': comparison['탑승건수_제공'].fillna(0).sum(),
    '총합 차이(정제-제공)': comparison['차이(정제-제공)'].sum(),
    '완전 일치 행 수': (comparison['차이(정제-제공)'] == 0).sum(),
    '불일치 행 수': (comparison['차이(정제-제공)'] != 0).sum(),
    '제공 데이터 누락 시간대': (comparison['_merge'] == 'left_only').sum(),
})
display(comparison_summary.to_frame('값'))
comparison.loc[comparison['차이(정제-제공)'] != 0].head(20)

## 7. 정제 CSV 저장

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_output = df_boarding_hour.copy()
df_output['일자'] = df_output['일자'].dt.strftime('%Y-%m-%d')
df_output.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')

print(f'저장 완료: {OUTPUT_PATH}')
print(f'저장 행 수: {len(df_output):,}')
print(f"총 탑승건수: {df_output['탑승건수'].sum():,}")
df_output.head(24)